# DEE — Ecuación de estado de defectos blandos

**Pregunta central:** ¿el modo localizado de un defecto blando 3D tiene parámetro de ecuación de estado w ≈ -1, lo que lo haría candidato a fuente de Λ?

**Contexto:** en notebooks anteriores mostramos:
- El vacío del cristal homogéneo tiene w ≈ +0.07 (no es Λ)
- Los defectos blandos 3D capturan estados ligados con energías por debajo del continuo

**Hipótesis:** el modo localizado, al estar atrapado en un pozo elástico, tiene una ecuación de estado distinta al cristal homogéneo. Si la presión efectiva del modo es negativa, sería análogo a una vacuum bubble cosmológica.

**Método:**
1. Construir cristal con inclusión 3D blanda (factor ×0.1)
2. Identificar el modo localizado de menor frecuencia
3. Comprimir cristal levemente (5 valores de ε)
4. Trackear cómo cambia ω_modo y la densidad efectiva del modo
5. Calcular w_defecto = P/(ρc²)

**Resultados posibles:**
- w ≈ -1: defectos son candidato fuerte para Λ
- w ≈ +0.07: defectos no aportan más que el cristal homogéneo
- w intermedio: comportamiento híbrido a interpretar

---

In [ ]:
import os
import numpy as np
from scipy.sparse import csr_matrix, diags
from scipy.sparse.linalg import eigsh
import matplotlib.pyplot as plt
import time

PLOT_DIR = 'plots_w_defecto'
os.makedirs(PLOT_DIR, exist_ok=True)

def save_plot(name, fig=None, dpi=120):
    if fig is None: fig = plt.gcf()
    path = os.path.join(PLOT_DIR, f'{name}.png')
    fig.savefig(path, dpi=dpi, bbox_inches='tight')
    print(f'  → {path}')

print('Setup listo')

In [ ]:
# Funciones del cristal con defectos
def periodic_distance_matrix(points, L=1.0):
    diff = points[:, None, :] - points[None, :, :]
    diff = diff - L * np.round(diff / L)
    return np.linalg.norm(diff, axis=-1)

def grid_perturbed_T3(N_target, jitter_fraction, seed=42):
    rng = np.random.default_rng(seed)
    n_side = round(N_target**(1/3))
    N_actual = n_side**3
    spacing = 1.0 / n_side
    coords = np.arange(n_side) * spacing + spacing/2
    gx, gy, gz = np.meshgrid(coords, coords, coords, indexing='ij')
    points = np.stack([gx.ravel(), gy.ravel(), gz.ravel()], axis=1)
    points += rng.uniform(-jitter_fraction*spacing, jitter_fraction*spacing, size=points.shape)
    points = points % 1.0
    return points, N_actual

def build_laplacian_with_defect(points, k_neighbors=30, alpha=2.0, L=1.0,
                                  force_modifiers=None):
    """Laplaciano con factores de fuerza modificados en algunos nodos (defectos)."""
    D_mat = periodic_distance_matrix(points, L=L)
    N = len(points)
    if force_modifiers is None: force_modifiers = {}
    D_search = D_mat.copy()
    np.fill_diagonal(D_search, np.inf)
    neighbor_idx = np.argpartition(D_search, k_neighbors, axis=1)[:, :k_neighbors]
    rows, cols, vals = [], [], []
    for i in range(N):
        for j in neighbor_idx[i]:
            d = D_mat[i, j]
            if d > 0:
                w = 1.0 / d**alpha
                mod_i = force_modifiers.get(i, 1.0)
                mod_j = force_modifiers.get(j, 1.0)
                w = w * np.sqrt(mod_i * mod_j)
                rows.append(i); cols.append(j); vals.append(w)
    W = csr_matrix((vals, (rows, cols)), shape=(N, N))
    W = (W + W.T) / 2
    degrees = np.array(W.sum(axis=1)).flatten()
    return diags(degrees) - W, D_mat

def participation_ratio(eigvec):
    v = eigvec / np.linalg.norm(eigvec)
    v2 = v**2
    ipr = np.sum(v2**2)
    return 1.0 / ipr if ipr > 0 else len(v)

# Cristal de referencia (sin defecto)
N_TARGET = 1331
JITTER = 0.1
K_NEIGHBORS = 30

points_base, N = grid_perturbed_T3(N_TARGET, JITTER, seed=42)

# Construir defecto blando
center = np.array([0.5, 0.5, 0.5])
diffs = points_base - center; diffs -= np.round(diffs)
dist_center = np.linalg.norm(diffs, axis=1)
INCLUSION_RADIUS = 0.15
INCL_SOFT = 0.1

inclusion_nodes = np.where(dist_center < INCLUSION_RADIUS)[0]
force_mods = {int(i): INCL_SOFT for i in inclusion_nodes}

print(f'Configuración:')
print(f'  N = {N} nodos')
print(f'  Defecto: inclusión esférica de radio {INCLUSION_RADIUS}')
print(f'  Nodos en defecto: {len(inclusion_nodes)} ({len(inclusion_nodes)/N*100:.1f}%)')
print(f'  Factor de blandura: ×{INCL_SOFT}')

## Paso 1 — Construir cristales con defecto a distintos volúmenes

Para extraer presión, necesitamos E_modo(V) en varios V cercanos. Aplicamos compresión/expansión uniforme isotrópica:

In [ ]:
epsilons = [-0.02, -0.01, 0.0, 0.01, 0.02]
results = []

print('Calculando E_modo(V) para defecto blando...')
print()

for eps in epsilons:
    t0 = time.time()
    L_box = 1.0 + eps
    points_eps = points_base * L_box  # los puntos se reescalan junto con la caja
    
    L_op_eps, D_eps = build_laplacian_with_defect(
        points_eps, K_NEIGHBORS, alpha=2.0, L=L_box,
        force_modifiers=force_mods
    )
    
    # Calcular suficientes autovalores para capturar bien los modos localizados
    eigs_eps, vecs_eps = eigsh(L_op_eps, k=80, which='SM', tol=1e-8)
    idx_s = np.argsort(eigs_eps)
    eigs_eps = eigs_eps[idx_s]
    vecs_eps = vecs_eps[:, idx_s]
    
    # Filtrar modo cero
    mask_nz = eigs_eps > 1e-3
    eigs_nz = eigs_eps[mask_nz]
    vecs_nz = vecs_eps[:, mask_nz]
    omegas_eps = np.sqrt(eigs_nz)
    
    # Calcular participation ratios para identificar el modo más localizado
    parts = np.array([participation_ratio(vecs_nz[:, i]) for i in range(len(omegas_eps))])
    
    # El modo más localizado (menor participation) entre los primeros 30
    idx_localized = int(np.argmin(parts[:30]))
    omega_loc = omegas_eps[idx_localized]
    part_loc = parts[idx_localized]
    vec_loc = vecs_nz[:, idx_localized]
    
    # Energía del modo localizado (½ħω en unidades naturales)
    E_modo_loc = 0.5 * omega_loc
    
    # Volumen efectivo del modo localizado
    # Definimos V_modo como el volumen "cubierto" por |v|² del modo
    # V_modo = participation_ratio · volumen_por_nodo = part · V_total / N
    V_total = L_box**3
    V_modo = part_loc * V_total / N
    
    # Densidad de energía del modo localizado
    rho_modo = E_modo_loc / V_modo
    
    # Frecuencia mínima para reportar
    omega_min = omegas_eps[0]
    
    results.append({
        'eps': eps,
        'V_total': V_total,
        'V_modo': V_modo,
        'omega_loc': omega_loc,
        'omega_min': omega_min,
        'part_loc': part_loc,
        'idx_loc': idx_localized,
        'E_modo': E_modo_loc,
        'rho_modo': rho_modo,
        'time': time.time() - t0
    })
    
    r = results[-1]
    print(f'  ε={eps:+.3f}: V={V_total:.4f}, ω_loc={omega_loc:.3f}, '
          f'part={part_loc:.1f}, E_modo={E_modo_loc:.4f}, '
          f'tiempo={r["time"]:.0f}s')

## Paso 2 — Cálculo de la ecuación de estado del modo localizado

Tres definiciones alternativas de presión que nos dan info distinta:

**Definición 1 — Presión total del cristal con defecto:**  
P_total = -∂E_total/∂V_total

**Definición 2 — Presión del modo localizado en su volumen efectivo:**  
P_modo = -∂E_modo/∂V_modo

**Definición 3 — Presión cosmológica (relevante para Λ):**  
Si el modo está confinado en un volumen fijo (no se expande con el cristal), tiene presión efectiva 0 cuando el universo se expande. La pregunta es cómo cambia su energía total con la expansión cósmica.

La definición físicamente más relevante para Λ es la **Definición 3**: cuando el universo se expande, ¿el modo localizado se diluye o no?

In [ ]:
# Arrays
eps_arr = np.array([r['eps'] for r in results])
V_total_arr = np.array([r['V_total'] for r in results])
V_modo_arr = np.array([r['V_modo'] for r in results])
omega_loc_arr = np.array([r['omega_loc'] for r in results])
E_modo_arr = np.array([r['E_modo'] for r in results])
rho_modo_arr = np.array([r['rho_modo'] for r in results])
part_arr = np.array([r['part_loc'] for r in results])

idx_0 = list(eps_arr).index(0.0)

# DEFINICIÓN 1: Presión del modo en su volumen efectivo
# Si V_modo cambia al comprimir el cristal, ¿cómo cambia E_modo?
dE_dVmodo = (E_modo_arr[idx_0+1] - E_modo_arr[idx_0-1]) / (V_modo_arr[idx_0+1] - V_modo_arr[idx_0-1])
P_modo_dVmodo = -dE_dVmodo

# Velocidad del sonido (del cristal homogéneo, ya conocida)
c_s = 2.186  # de los notebooks anteriores

rho_modo_central = rho_modo_arr[idx_0]
w_definicion_1 = P_modo_dVmodo / (rho_modo_central * c_s**2)

# DEFINICIÓN 2: Presión cosmológica relevante
# Si V_total cambia (universo expande), ¿cómo cambia E_modo?
# Esta es la pregunta clave para Λ
dE_dVtotal = (E_modo_arr[idx_0+1] - E_modo_arr[idx_0-1]) / (V_total_arr[idx_0+1] - V_total_arr[idx_0-1])
P_cosmologica = -dE_dVtotal

# Densidad efectiva en volumen total (no en V_modo)
rho_efectiva_cosmologica = E_modo_arr[idx_0] / V_total_arr[idx_0]
w_cosmologica = P_cosmologica / (rho_efectiva_cosmologica * c_s**2)

# DEFINICIÓN 3: Comparar con cristal homogéneo
# El cristal homogéneo dio w ≈ +0.07 (P > 0)
# Para que el defecto sea fuente de Λ, su w cosmológico debe ser cercano a -1

print('='*70)
print('ANÁLISIS DE ECUACIÓN DE ESTADO DEL MODO LOCALIZADO')
print('='*70)
print()
print('Definición 1: presión interna del modo en su volumen efectivo')
print(f'  ∂E_modo/∂V_modo  = {dE_dVmodo:+.4f}')
print(f'  P_interna_modo   = {P_modo_dVmodo:+.4f}')
print(f'  ρ_modo (en V_modo) = {rho_modo_central:.4f}')
print(f'  w_interna        = {w_definicion_1:+.4f}')
print()
print('Definición 2: presión cosmológica (la que importa para Λ)')
print(f'  ∂E_modo/∂V_total = {dE_dVtotal:+.6f}')
print(f'  P_cosmológica    = {P_cosmologica:+.6f}')
print(f'  ρ_efectiva (en V_total) = {rho_efectiva_cosmologica:.6f}')
print(f'  w_cosmológico    = {w_cosmologica:+.4f}')
print()
print('Comparación con benchmarks cosmológicos:')
print(f'  Cristal homogéneo:  w = +0.07')
print(f'  Polvo (materia):    w =  0.00')
print(f'  Constante cosmológica (Λ): w = -1.00')

In [ ]:
# Plots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Panel 1: ω_loc vs V_total
ax = axes[0, 0]
ax.plot(V_total_arr, omega_loc_arr, 'o-', markersize=12, color='crimson', lw=2)
ax.set_xlabel('V_total (volumen del universo)', fontsize=12)
ax.set_ylabel('ω_modo (frecuencia del modo localizado)', fontsize=12)
ax.set_title('Frecuencia del modo localizado vs volumen', fontsize=12)
ax.grid(alpha=0.3)

# Panel 2: E_modo vs V_total
ax = axes[0, 1]
ax.plot(V_total_arr, E_modo_arr, 'o-', markersize=12, color='darkgreen', lw=2,
        label='E_modo medido')
# Ajuste cuadrático
coefs = np.polyfit(V_total_arr, E_modo_arr, 2)
V_smooth = np.linspace(V_total_arr.min(), V_total_arr.max(), 100)
ax.plot(V_smooth, np.polyval(coefs, V_smooth), '-', color='red', lw=1.5, alpha=0.7,
        label='Ajuste cuadrático')
ax.set_xlabel('V_total', fontsize=12)
ax.set_ylabel('E_modo (energía del modo localizado)', fontsize=12)
ax.set_title('E_modo(V_total) — pendiente determina P cosmológica', fontsize=12)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

# Panel 3: comparación de w con benchmarks
ax = axes[1, 0]
categories = ['Radiación\n(w=1/3)', 'Cristal\nhomogéneo\n(w=0.07)', 
              'Polvo\n(w=0)', 'Frontera\n(w=-1/3)',
              'Λ\n(w=-1)', f'DEE defecto\nblando\n(w={w_cosmologica:+.2f})']
values = [1/3, 0.07, 0, -1/3, -1, w_cosmologica]
colors = ['orange', 'steelblue', 'gray', 'purple', 'green', 'crimson']
ax.barh(categories, values, color=colors, alpha=0.7)
ax.axvline(0, color='black', lw=1)
ax.axvline(-1, color='green', linestyle='--', alpha=0.5, label='Λ requerido')
ax.set_xlabel('Parámetro de ecuación de estado w', fontsize=12)
ax.set_title('w del defecto blando vs benchmarks cosmológicos', fontsize=12)
ax.legend(fontsize=10)
ax.grid(alpha=0.3, axis='x')

# Panel 4: tracking del modo (para ver si seguimos el mismo modo en todos los V)
ax = axes[1, 1]
ax.plot(V_total_arr, part_arr, 'o-', markersize=12, color='darkblue', lw=2)
ax.axhline(N/10, color='red', linestyle='--', label='Umbral localización')
ax.set_xlabel('V_total', fontsize=12)
ax.set_ylabel('Participation ratio del modo', fontsize=12)
ax.set_title('Tracking: ¿seguimos el mismo modo localizado?', fontsize=12)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)

plt.tight_layout()
save_plot('40_w_defecto_blando')
plt.show()

In [ ]:
# Análisis adicional: estado ligado y dilución
# Para Λ: la densidad de energía debe NO diluirse con la expansión
# ρ_Λ permanece constante, mientras que ρ_materia ∝ 1/a³ y ρ_radiación ∝ 1/a⁴

# Para el modo localizado: ρ_modo = E_modo / V_total
# Si E_modo permanece constante con V → ρ_modo se diluye como 1/V (peor que materia)
# Si E_modo crece con V → ρ_modo podría no diluirse

print('Análisis de dilución del estado ligado con la expansión:')
print()
print(f'{"V_total":>10} {"E_modo":>10} {"E_modo · V":>14} {"E_modo · V²":>14} {"ρ_modo":>14}')
print('-'*65)
for r in results:
    rho_efec = r['E_modo'] / r['V_total']
    EV = r['E_modo'] * r['V_total']
    EV2 = r['E_modo'] * r['V_total']**2
    print(f'{r["V_total"]:>10.4f} {r["E_modo"]:>10.4f} '
          f'{EV:>14.6f} {EV2:>14.6f} {rho_efec:>14.6e}')

# Para Λ verdadero: E_modo ∝ V_total (porque ρ es constante → E = ρV)
# Verificamos pendiente de E vs V
from scipy.stats import linregress
slope_EV, intercept_EV, r_EV, _, _ = linregress(V_total_arr, E_modo_arr)
print()
print(f'Ajuste lineal E_modo = a + b·V:')
print(f'  pendiente b = {slope_EV:.4f}')
print(f'  intercepto a = {intercept_EV:.4f}')
print(f'  ratio b·V_central/E_total = {slope_EV * V_total_arr[idx_0] / E_modo_arr[idx_0]:.4f}')
print()
print('Interpretación:')
if slope_EV > 0 and abs(intercept_EV / E_modo_arr[idx_0]) < 0.1:
    print('  → E ∝ V con intercepto pequeño: comportamiento tipo Λ')
elif abs(slope_EV) < 1e-3:
    print('  → E aproximadamente constante: ρ se diluye con V')
    print('  → Comportamiento tipo "materia localizada" más que Λ')
else:
    print('  → Comportamiento intermedio o no estándar')

In [ ]:
print('='*70)
print('VEREDICTO — ¿Es el defecto blando candidato a Λ?')
print('='*70)
print()
print(f'Resultado central:')
print(f'  w cosmológico del defecto blando: {w_cosmologica:+.4f}')
print(f'  w requerido para Λ:               -1.0000')
print(f'  Diferencia:                       {abs(w_cosmologica - (-1)):.4f}')
print()

if abs(w_cosmologica + 1) < 0.2:
    print('  → ✓✓ COMPATIBLE CON Λ')
    print('     El modo localizado del defecto blando 3D tiene ecuación de estado')
    print('     cercana a w = -1, característica de constante cosmológica.')
    print('     Una densidad uniforme de tales defectos podría reproducir Λ_obs.')
elif abs(w_cosmologica) < 0.2:
    print('  → ~ COMPORTAMIENTO TIPO POLVO')
    print('     El defecto se comporta como materia no relativista, no como Λ.')
    print('     w cercano a 0 indica energía localizada que se diluye con V.')
elif w_cosmologica > 0.5:
    print('  → ~ COMPORTAMIENTO TIPO RADIACIÓN')
    print('     w positivo grande indica presión positiva fuerte, no es Λ.')
elif -0.5 < w_cosmologica < -0.2:
    print('  → ~ COMPORTAMIENTO INTERMEDIO')
    print('     w negativo pero no -1: contribución parcial a aceleración cósmica')
    print('     pero no es Λ pura.')
else:
    print(f'  → COMPORTAMIENTO INESPERADO (w = {w_cosmologica:+.2f})')
    print('     Requiere análisis adicional.')

print()
print('Implicación para DEE:')
if abs(w_cosmologica + 1) < 0.2:
    print('  Los defectos blandos 3D del cristal son CANDIDATOS VIABLES para Λ.')
    print('  Esto resuelve la pregunta abierta sobre qué objeto en DEE puede')
    print('  ser fuente de la constante cosmológica observada.')
elif abs(w_cosmologica) < 0.2:
    print('  Los defectos se comportan como MATERIA LOCALIZADA, no como Λ.')
    print('  Esto es consistente con la conjetura de defectos = materia oscura,')
    print('  pero deja abierta la pregunta sobre el origen de Λ en DEE.')
else:
    print('  El comportamiento del defecto difiere tanto del cristal homogéneo (w=0.07)')
    print('  como de Λ pura (w=-1). Requiere interpretación adicional.')

In [ ]:
import shutil
shutil.make_archive('plots_w_defecto', 'zip', PLOT_DIR)
print(f'\nZip creado: plots_w_defecto.zip')

try:
    from google.colab import files
    files.download('plots_w_defecto.zip')
    print('→ Descarga iniciada')
except ImportError:
    pass